# Evaluate a Trained Checkpoint

Load a checkpoint, run N evaluation episodes in `CubeReachV1`, and display the `EvalResult`.

Set `CHECKPOINT_PATH` and `N_EPISODES` in Cell 1, then run all cells.

In [ ]:
# Cell 1 — Configuration
CHECKPOINT_PATH = "../checkpoints/checkpoint_00010000.ckpt"  # adjust as needed
N_EPISODES = 50
POLICY = "act"  # "act" or "diffusion"
DEVICE = "cpu"  # "cpu" or "cuda"

In [ ]:
# Cell 2 — Load policy from checkpoint
from pathlib import Path
from omegaconf import OmegaConf

from playground.policies.act_wrapper import ACTWrapper
from playground.policies.diffusion_wrapper import DiffusionWrapper

# Load the policy config bundled alongside the checkpoint (saved by Trainer)
ckpt_path = Path(CHECKPOINT_PATH)
cfg_path = ckpt_path.parent / "config.yaml"
policy_cfg = OmegaConf.load(cfg_path).policy if cfg_path.exists() else OmegaConf.create({
    "name": POLICY,
    "pretrained_repo": None,
    "n_obs_steps": 1,
    "n_action_steps": 100,
    "chunk_size": 100,
    "input_shapes": {"observation.state": [3]},
    "output_shapes": {"action": [8]},
})

PolicyClass = ACTWrapper if POLICY == "act" else DiffusionWrapper
policy = PolicyClass(policy_cfg, device=DEVICE)
policy.load_checkpoint(ckpt_path)
print(f"Loaded {POLICY} checkpoint from {ckpt_path.name}")

In [ ]:
# Cell 3 — Run evaluation
from playground.eval.evaluator import Evaluator

evaluator = Evaluator(
    policy=policy,
    env_name="CubeReachV1",
    n_episodes=N_EPISODES,
)
result = evaluator.evaluate()

print(f"\n{'='*40}")
print(f"Episodes      : {result.n_episodes}")
print(f"Success rate  : {result.success_rate:.1%}")
print(f"Mean reward   : {result.mean_reward:.3f}")
print(f"{'='*40}")

In [ ]:
# Cell 4 — Plot per-episode rewards
import matplotlib.pyplot as plt
import numpy as np

rewards = result.per_episode_rewards
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(rewards)), rewards, color="steelblue", alpha=0.7)
ax.axhline(result.mean_reward, color="red", linewidth=2, label=f"mean={result.mean_reward:.2f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Cumulative Reward")
ax.set_title(f"{POLICY.upper()} — {N_EPISODES} eval episodes (success={result.success_rate:.1%})")
ax.legend()
plt.tight_layout()
plt.show()